In [3]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

load_dotenv(override=True)

True

#### A first model call  
`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.  
We call `invoke` with a prompt; this is a key method in LangChain.

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini')

message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"
reply = llm.invoke(message)
display(Markdown(reply.content))

An autonomous AI agent is capable of performing tasks and making decisions independently without human intervention, using its own reasoning and adaptive learning processes.

#### Streaming
For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [6]:
for chunk in llm.stream("Tell me a two line poem about atonoumus agents"):
    print(chunk.content, end='', flush=True)

In realms of code where thoughts reside,  
Autonomous agents learn and guide.

#### Messages
LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [7]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?")
]

print(llm.invoke(messages).content)

Paris is the capital city.


In [8]:
# The exact same call using plain dictionaries,
messages_as_dict = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"}
]

print(llm.invoke(messages_as_dict).content)

Paris is the capital of France.


#### Tools with the `@tool` decorator
A tool is a Python function the model is allowed to call.  
The modern way to make one is the @tool decorator.  
Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.  

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [16]:
import yfinance as yf

@tool 
def get_share_price(ticker_symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    ticker = yf.Ticker(ticker_symbol)
    historical_data = ticker.history(period="1d")
    closing_price = historical_data['Close'].iloc[-1]
    return closing_price

In [17]:
# The first step is to bind the tools to the model with bind_tools. 
# Now when we invoke, the model may come back not with an answer but with a request to run a tool. 
# That request shows up in .tool_calls.

llm_with_tools = llm.bind_tools([get_share_price])
response = llm_with_tools.invoke("What is the share price of Amazon?")

print(f"Content: {response.content}")
print(f"tool calls: {response.tool_calls}")

Content: 
tool calls: [{'name': 'get_share_price', 'args': {'ticker_symbol': 'AMZN'}, 'id': 'call_XJ0Z2CEQhUxXjAeLoOgnt7dU', 'type': 'tool_call'}]


In [18]:
# Running the tool loop by hand
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call['name'] == 'get_share_price':
        result = get_share_price.invoke(call['args'])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call['id']))

# Invoke again, now that model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

The current share price of Amazon (AMZN) is approximately $249.67.


#### Structured output

In [19]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="Stock ticker symbol")
    founded_year: int = Field(description="The year company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke('Tell me about Amazon the technology company')

print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN
